##### How to read `singlelineList` in pyspark?

- The **entire file** is on **one line**.
- The **root element** is a **JSON array ([ {…}, {…} ])**.

     [{"id":1,"name":"A","price":100},{"id":2,"name":"B","price":200}]

**Content**
- `Using multiLine = true (Recommended)`
- `What happens if you don’t use multiLine=true?`
- `Explicit schema`
- `How to read multiple singlelineList files?`
- `PERMISSIVE mode & explicit schema without _corrupt_record`

##### 1) Using multiLine = true (Recommended)
- Even though it’s **one line**, Spark needs **multiLine=true** because the **root is an array**.
- Works for:
  - **Single line JSON array (SingleLine List)**.
  - **Multi-line JSON array**.

In [0]:
path = "/Volumes/azureadb/pyspark/training/json/read_json_singlelineList_same_schema/singlelineList_01.json"

df_singList_mlt = spark.read \
    .format("json")\
    .option('multiLine', True)\
    .load(path)

display(df_singList_mlt)
df_singList_mlt.printSchema()

country,description,id,input_timestamp,last_update_timestamp,source,user
IND,SQLServer,1,1124256609,1524256609,SQLDB,Jagan
US,ABAP,2,1224256609,1424256609,SAP,Brindavan
CANADA,GEN2,3,1324256609,1524256609,ADLS,Nandu
US,Storage,4,1424256609,1724256609,Blob,Syamala
SWEDEN,Lake House,5,1524256609,1664256609,Data Lake Storage,Chethan
UK,Lake Warehouse,6,1624256609,1874256609,Delta Lake,Roopesh
Norway,OracleDB,7,1779256609,188256609,oracle,Sundar
SWEDEN,ML,8,1524256609,1664256609,DS,Swaroop
UK,Machine,9,1924256609,1674256609,MLOPS,Rahul
Norway,DataScience,10,1379256609,198256609,AI,Santhosh


root
 |-- country: string (nullable = true)
 |-- description: string (nullable = true)
 |-- id: long (nullable = true)
 |-- input_timestamp: long (nullable = true)
 |-- last_update_timestamp: long (nullable = true)
 |-- source: string (nullable = true)
 |-- user: string (nullable = true)



##### 2) What happens if you don’t use multiLine=true?

- **Entire row becomes NULL**.
- Or **_corrupt_record** is populated.
- Because Spark expects **one JSON object per line**, **not a list**.
  - `spark.read.json("/path/to/singleline_list.json")`

In [0]:
df_singList = spark.read \
    .format("json") \
    .load(path)
display(df_singList)

_corrupt_record,country,description,id,input_timestamp,last_update_timestamp,source,user
null,IND,SQLServer,1,1124256609,1524256609,SQLDB,Jagan
null,US,ABAP,2,1224256609,1424256609,SAP,Brindavan
null,CANADA,GEN2,3,1324256609,1524256609,ADLS,Nandu
null,US,Storage,4,1424256609,1724256609,Blob,Syamala
null,SWEDEN,Lake House,5,1524256609,1664256609,Data Lake Storage,Chethan
null,UK,Lake Warehouse,6,1624256609,1874256609,Delta Lake,Roopesh
null,Norway,OracleDB,7,1779256609,188256609,oracle,Sundar
null,SWEDEN,ML,8,1524256609,1664256609,DS,Swaroop
null,UK,Machine,9,1924256609,1674256609,MLOPS,Rahul
null,Norway,DataScience,10,1379256609,198256609,AI,Santhosh


##### 3) Explicit schema (Best practice)

- **Faster** than `inference`.
- **Prevents datatype / schema inference issues**.

In [0]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType, LongType

# Define the main schema including the nested structure
Schema = StructType([StructField('id', IntegerType(), False),
                     StructField('country', StringType(), False),
                     StructField('description', StringType(), False),
                     StructField('input_timestamp', LongType(), False),
                     StructField('last_update_timestamp', LongType(), False),
                     StructField('source', StringType(), False),
                     StructField('user', StringType(), False)]
                     )

df_singList_mlt_schema = spark.read.format("json")\
                                   .option('multiLine', True)\
                                   .schema(Schema)\
                                   .json(path)

display(df_singList_mlt_schema)
df_singList_mlt_schema.printSchema()

id,country,description,input_timestamp,last_update_timestamp,source,user
1,IND,SQLServer,1124256609,1524256609,SQLDB,Jagan
2,US,ABAP,1224256609,1424256609,SAP,Brindavan
3,CANADA,GEN2,1324256609,1524256609,ADLS,Nandu
4,US,Storage,1424256609,1724256609,Blob,Syamala
5,SWEDEN,Lake House,1524256609,1664256609,Data Lake Storage,Chethan
6,UK,Lake Warehouse,1624256609,1874256609,Delta Lake,Roopesh
7,Norway,OracleDB,1779256609,188256609,oracle,Sundar
8,SWEDEN,ML,1524256609,1664256609,DS,Swaroop
9,UK,Machine,1924256609,1674256609,MLOPS,Rahul
10,Norway,DataScience,1379256609,198256609,AI,Santhosh


root
 |-- id: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- description: string (nullable = true)
 |-- input_timestamp: long (nullable = true)
 |-- last_update_timestamp: long (nullable = true)
 |-- source: string (nullable = true)
 |-- user: string (nullable = true)



In [0]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType, LongType

# Define the main schema including the nested structure
Schema = StructType([StructField('id', IntegerType(), False),
                     StructField('source', StringType(), False),
                     StructField('description', StringType(), False),
                     StructField('input_timestamp', LongType(), False),
                     StructField('last_update_timestamp', LongType(), False),
                     StructField('country', StringType(), False),
                     StructField('user', StringType(), False)]
                     )

df_singList_mlt_schema_01 = spark.read.format("json")\
                                   .option('multiLine', True)\
                                   .schema(Schema)\
                                   .json(path)

display(df_singList_mlt_schema_01)
df_singList_mlt_schema_01.printSchema()

id,source,description,input_timestamp,last_update_timestamp,country,user
1,SQLDB,SQLServer,1124256609,1524256609,IND,Jagan
2,SAP,ABAP,1224256609,1424256609,US,Brindavan
3,ADLS,GEN2,1324256609,1524256609,CANADA,Nandu
4,Blob,Storage,1424256609,1724256609,US,Syamala
5,Data Lake Storage,Lake House,1524256609,1664256609,SWEDEN,Chethan
6,Delta Lake,Lake Warehouse,1624256609,1874256609,UK,Roopesh
7,oracle,OracleDB,1779256609,188256609,Norway,Sundar
8,DS,ML,1524256609,1664256609,SWEDEN,Swaroop
9,MLOPS,Machine,1924256609,1674256609,UK,Rahul
10,AI,DataScience,1379256609,198256609,Norway,Santhosh


root
 |-- id: integer (nullable = true)
 |-- source: string (nullable = true)
 |-- description: string (nullable = true)
 |-- input_timestamp: long (nullable = true)
 |-- last_update_timestamp: long (nullable = true)
 |-- country: string (nullable = true)
 |-- user: string (nullable = true)



##### 4) How to read multiple singlelineList files?
- Directory Reading
- If you have **multiple single-line JSON files** stored in **one folder**, pass the **directory path** instead of a file name to load them all into a **single DataFrame** simultaneously.

In [0]:
path01 = "/Volumes/azureadb/pyspark/training/json/read_json_singlelineList_same_schema/singlelineList_01.json"
path02 = "/Volumes/azureadb/pyspark/training/json/read_json_singlelineList_same_schema/singlelineList_02.json"
path03 = "/Volumes/azureadb/pyspark/training/json/read_json_singlelineList_same_schema/singlelineList_03.json"

df_singList_mlt_path = spark.read.format("json")\
                                 .option('multiLine', True)\
                                 .json([path01, path02, path03])

display(df_singList_mlt_path)

country,description,id,input_timestamp,last_update_timestamp,source,user
IND,SQLServer,21,1124256609,1524256609,SQLDB,Jaleel
US,ABAP,22,1224256609,1424256609,SAP,Bhavya
CANADA,GEN2,23,1324256609,1524256609,ADLS,Nirmal
US,Storage,24,1424256609,1724256609,Blob,Kavitha
SWEDEN,Lake House,25,1524256609,1664256609,Data Lake Storage,Guptha
UK,Lake Warehouse,26,1624256609,1874256609,Delta Lake,Deepak
Norway,OracleDB,27,1779256609,188256609,oracle,Sindu
SWEDEN,ML,28,1524256609,1664256609,DS,Yamuna
UK,Machine,29,1924256609,1674256609,MLOPS,Gajendra
Norway,DataScience,30,1379256609,198256609,AI,Yunus


In [0]:
# Read all multiple single line files
df_all_singList_mlt_path = spark.read \
    .option("multiline","true") \
    .json("/Volumes/azureadb/pyspark/training/json/read_json_singlelineList_same_schema/*.json")

display(df_all_singList_mlt_path)
df_all_singList_mlt_path.printSchema()

country,description,id,input_timestamp,last_update_timestamp,source,user
IND,SQLServer,21,1124256609,1524256609,SQLDB,Jaleel
US,ABAP,22,1224256609,1424256609,SAP,Bhavya
CANADA,GEN2,23,1324256609,1524256609,ADLS,Nirmal
US,Storage,24,1424256609,1724256609,Blob,Kavitha
SWEDEN,Lake House,25,1524256609,1664256609,Data Lake Storage,Guptha
UK,Lake Warehouse,26,1624256609,1874256609,Delta Lake,Deepak
Norway,OracleDB,27,1779256609,188256609,oracle,Sindu
SWEDEN,ML,28,1524256609,1664256609,DS,Yamuna
UK,Machine,29,1924256609,1674256609,MLOPS,Gajendra
Norway,DataScience,30,1379256609,198256609,AI,Yunus


root
 |-- country: string (nullable = true)
 |-- description: string (nullable = true)
 |-- id: long (nullable = true)
 |-- input_timestamp: long (nullable = true)
 |-- last_update_timestamp: long (nullable = true)
 |-- source: string (nullable = true)
 |-- user: string (nullable = true)



In [0]:
# Read all multiple single line files
df_all_singList_mlt_path_mixed_root = spark.read \
    .option("multiline","true") \
    .json("/Volumes/azureadb/pyspark/training/mixed_files/")

display(df_all_singList_mlt_path_mixed_root)
df_all_singList_mlt_path_mixed_root.printSchema()

First_Name Last_Name S.No _corrupt_record country description input_timestamp last_update_timestamp source null null null S.No,Status,Start_Date,End_Date,Expiration_Date,Last_Date,Input_Start_Date,Input_End_Date,Delivery_Start_Date,Delivery_End_Date,Delivery_Last_Date,Pricing_Date
1,Sell,26-Jan-24,31-03-2024,26-02-2024,26-Jan-24,11-03-2024,26-02-2024,21-12-2023,01-03-2024,02-04-2024,26-Jan-24
2,Buy,21-Dec-23,31-01-2024,21-12-2023,21-Dec-23,11-12-2024,27-12-2023,28-11-2024,01-01-2024,31-01-2024,21-Dec-23
3,Buy,27-Mar-25,30-06-2025,27-03-2025,27-Mar-25,21-04-2025,27-12-2024,26-02-2024,01-04-2025,30-06-2025,27-Mar-25
4,Sell,27-Dec-23,31-12-2024,27-12-2023,27-Dec-23,16-07-2024,27-12-2023,27-12-2023,01-01-2024,27-12-2023,27-Dec-23
5,Buy,29-Apr-24,30-04-2024,29-04-2024,29-Apr-24,22-04-2024,26-03-2024,09-01-2023,01-04-2024,30-04-2024,29-Apr-24
6,Buy,27-Dec-24,31-12-2025,27-12-2024,27-Dec-24,01-01-2025,21-12-2023,21-02-2023,01-01-2025,31-12-2025,27-Dec-24
7,Sell,21-Dec-23,28-02-2023,27-12-2023,21-Dec-23,01-02-2023,28-11-2024,29-05-2023,01-02-2023,20-03-2023,21-Dec-23
8,Buy,26-Mar-24,30-06-2024,26-03-2024,26-Mar-24,01-02-2023,26-02-2024,19-05-1985,01-02-2023,20-03-2023,26-Mar-24
9,Buy,27-Dec-23,28-02-2023,21-12-2023,27-Dec-23,21-04-2024,27-12-2023,27-04-2023,01-04-2024,26-03-2024,27-Dec-23
10,Buy,26-Jan-24,29-11-2024,28-11-2024,26-Jan-24,21-02-2023,01-02-2023,17-04-2020,01-02-2023,20-03-2023,26-Jan-24
11,Sell,13-Dec-23,30-06-2023,26-02-2024,13-Dec-23,11-02-2023,01-02-2023,28-04-2021,01-02-2023,20-03-2023,13-Dec-23
12,Buy,18-Mar-25,31-12-2024,27-12-2023,18-Mar-25,01-11-2024,01-11-2024,24-10-2024,01-11-2024,29-11-2024,18-Mar-25
13,Sell,27-Dec-23,31-12-2024,15-05-2023,27-Dec-23,01-04-2023,01-04-2023,01-04-2023,01-04-2023,22-05-2023,27-Dec-23
14,Buy,29-Apr-24,31-07-2023,15-05-2023,29-Apr-24,15-03-2023,01-05-2023,01-05-2023,01-05-2023,20-06-2023,29-Apr-24
15,Buy,21-Dec-24,30-06-2023,27-12-2023,21-Dec-24,01-01-2024,01-06-2023,01-06-2023,01-06-2023,20-07-2023,21-Dec-24
16,Buy,31-Dec-23,28-02-2023,27-09-2024,31-Dec-23,01-01-2024,01-04-2023,01-04-2023,01-04-2023,22-05-2023,31-Dec-23
17,Buy,12-Mar-24,31-07-2023,26-09-2024,12-Mar-24,29-04-2024,01-05-2023,26-02-2024,01-05-2023,20-06-2023,12-Mar-24
18,Sell,20-Dec-23,31-07-2023,28-06-2023,20-Dec-23,01-02-2023,01-06-2023,27-12-2023,01-06-2023,20-07-2023,20-Dec-23
19,Sell,26-Feb-24,30-06-2023,27-12-2023,26-Feb-24,01-02-2023,01-01-2024,27-12-2024,01-01-2024,27-12-2023,26-Feb-24
20,Sell,21-Oct-23,31-12-2024,27-02-2025,21-Oct-23,21-12-2023,01-01-2024,27-12-2023,01-01-2024,20-02-2024,21-Oct-23
21,Buy,27-May-25,31-12-2024,27-12-2024,27-May-25,28-11-2024,01-02-2024,26-03-2024,01-02-2024,20-03-2024,27-May-25
22,Sell,27-Dec-23,30-06-2023,27-12-2023,27-Dec-23,26-02-2024,01-03-2024,21-12-2023,01-03-2024,22-04-2024,27-Dec-23
23,Buy,29-Nov-24,31-12-2024,28-12-2023,29-Nov-24,27-12-2023,01-04-2024,28-11-2024,01-04-2024,20-05-2024,29-Nov-24
24,Buy,27-Dec-24,10-01-2023,13-03-2023,27-Dec-24,27-12-2024,01-05-2024,26-02-2024,01-05-2024,20-06-2024,27-Dec-24
25,Buy,21-Dec-23,28-02-2023,26-02-2024,21-Dec-23,27-12-2023,01-06-2024,27-12-2023,01-06-2024,22-07-2024,21-Dec-23
26,Buy,26-Jun-24,30-06-2023,21-12-2023,26-Jun-24,26-03-2024,01-07-2024,01-07-2024,01-07-2024,20-08-2024,26-Jun-24
27,Buy,27-Oct-23,31-12-2024,27-03-2025,27-Oct-23,21-12-2023,01-08-2024,01-08-2024,01-08-2024,20-09-2024,27-Oct-23
28,Buy,26-Jan-24,10-01-2023,27-12-2023,26-Jan-24,28-11-2024,01-09-2024,01-09-2024,01-09-2024,21-10-2024,26-Jan-24
29,Buy,13-Dec-23,28-02-2023,29-04-2024,13-Dec-23,26-02-2024,21-12-2023,01-10-2024,01-10-2024,20-11-2024,13-Dec-23
30,Buy,18-Mar-25,30-06-2023,27-12-2024,18-Mar-25,27-12-2023,28-11-2024,01-11-2024,01-11-2024,20-12-2024,18-Mar-25
31,Sell,27-Dec-23,28-02-2023,27-12-2023,27-Dec-23,15-05-2023,26-02-2024,01-12-2024,01-12-2024,20-01-2025,27-Dec-23
32,Sell,29-Aug-24,30-06-2023,26-03-2024,29-Aug-24,15-05-2023,27-12-2023,01-01-2024,01-01-2024,20-02-2024,29-Aug-24
33,Buy,21-Sep-24,31-12-2024,21-12-2

root
 |-- First_Name: string (nullable = true)
 |-- Last_Name: string (nullable = true)
 |-- S.No: long (nullable = true)
 |-- _corrupt_record: string (nullable = true)
 |-- country: string (nullable = true)
 |-- description: string (nullable = true)
 |-- input_timestamp: long (nullable = true)
 |-- last_update_timestamp: long (nullable = true)
 |-- source: string (nullable = true)



In [0]:
# Read all multiple single line files
df_all_singList_mlt_path_csv = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/azureadb/pyspark/training/mixed_files/*.csv")

display(df_all_singList_mlt_path_csv)
df_all_singList_mlt_path_csv.printSchema()

S.No,Status,Start_Date,End_Date,Expiration_Date,Last_Date,Input_Start_Date,Input_End_Date,Delivery_Start_Date,Delivery_End_Date,Delivery_Last_Date,Pricing_Date
1,Sell,26-Jan-24,2024-03-31,2024-02-26,26-Jan-24,2024-03-11,2024-02-26,2023-12-21,01-03-2024,2024-04-02,26-Jan-24
2,Buy,21-Dec-23,2024-01-31,2023-12-21,21-Dec-23,2024-12-11,2023-12-27,2024-11-28,01-01-2024,2024-01-31,21-Dec-23
3,Buy,27-Mar-25,2025-06-30,2025-03-27,27-Mar-25,2025-04-21,2024-12-27,2024-02-26,01-04-2025,2025-06-30,27-Mar-25
4,Sell,27-Dec-23,2024-12-31,2023-12-27,27-Dec-23,2024-07-16,2023-12-27,2023-12-27,01-01-2024,2023-12-27,27-Dec-23
5,Buy,29-Apr-24,2024-04-30,2024-04-29,29-Apr-24,2024-04-22,2024-03-26,2023-01-09,01-04-2024,2024-04-30,29-Apr-24
6,Buy,27-Dec-24,2025-12-31,2024-12-27,27-Dec-24,2025-01-01,2023-12-21,2023-02-21,01-01-2025,2025-12-31,27-Dec-24
7,Sell,21-Dec-23,2023-02-28,2023-12-27,21-Dec-23,2023-02-01,2024-11-28,2023-05-29,01-02-2023,2023-03-20,21-Dec-23
8,Buy,26-Mar-24,2024-06-30,2024-03-26,26-Mar-24,2023-02-01,2024-02-26,1985-05-19,01-02-2023,2023-03-20,26-Mar-24
9,Buy,27-Dec-23,2023-02-28,2023-12-21,27-Dec-23,2024-04-21,2023-12-27,2023-04-27,01-04-2024,2024-03-26,27-Dec-23
10,Buy,26-Jan-24,2024-11-29,2024-11-28,26-Jan-24,2023-02-21,2023-02-01,2020-04-17,01-02-2023,2023-03-20,26-Jan-24


root
 |-- S.No: integer (nullable = true)
 |-- Status: string (nullable = true)
 |-- Start_Date: string (nullable = true)
 |-- End_Date: date (nullable = true)
 |-- Expiration_Date: date (nullable = true)
 |-- Last_Date: string (nullable = true)
 |-- Input_Start_Date: date (nullable = true)
 |-- Input_End_Date: date (nullable = true)
 |-- Delivery_Start_Date: date (nullable = true)
 |-- Delivery_End_Date: string (nullable = true)
 |-- Delivery_Last_Date: date (nullable = true)
 |-- Pricing_Date: string (nullable = true)



##### 5) How to read `multiple singlelineList` files with `Null's & empty string`?

In [0]:
path = "/Volumes/azureadb/pyspark/training/json/read_json/SingleLine_List/singlelineList_null.json"

df_singList_null = spark.read \
    .format("json")\
    .option('multiLine', True)\
    .load(path)

display(df_singList_null)
df_singList_null.printSchema()

country,description,id,input_timestamp,last_update_timestamp,source,user
IND,SQLServer,1,null,1524256609,,Jagan
US,ABAP,2,1224256609,1424256609,SAP,Brindavan
CANADA,,3,1324256609,null,ADLS,Nandu
US,Storage,4,null,1724256609,,Syamala
SWEDEN,Lake House,5,null,1664256609,Data Lake Storage,Chethan
,,6,1624256609,null,Delta Lake,Roopesh
,OracleDB,7,null,188256609,oracle,Sundar
SWEDEN,ML,8,null,1664256609,,Swaroop
UK,Machine,9,1924256609,1674256609,MLOPS,Rahul
Norway,,null,1379256609,198256609,AI,


root
 |-- country: string (nullable = true)
 |-- description: string (nullable = true)
 |-- id: long (nullable = true)
 |-- input_timestamp: long (nullable = true)
 |-- last_update_timestamp: long (nullable = true)
 |-- source: string (nullable = true)
 |-- user: string (nullable = true)



##### 6) PERMISSIVE mode & explicit schema without `_corrupt_record`

In [0]:
df_singList_recrd_wo = spark.read \
    .format("json")\
    .option('multiLine', True)\
    .option("mode", "PERMISSIVE")\
    .load("/Volumes/azureadb/pyspark/training/json/read_json/SingleLine_List/singlelineList_corrupt_record.json")

display(df_singList_recrd_wo)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8018936799067135>, line 9
      1 from pyspark.sql.types import StructField, StructType, IntegerType, StringType, LongType
      3 df_singList_recrd_wo = spark.read \
      4     .format("json")\
      5     .option('multiLine', True)\
      6     .option("mode", "PERMISSIVE")\
      7     .load("/Volumes/azureadb/pyspark/training/json/read_json/SingleLine_List/singlelineList_corrupt_record.json")
----> 9 display(df_singList_recrd_wo)

File /databricks/python_shell/lib/dbruntime/display.py:133, in Display.display(self, input, *args, **kwargs)
    131     pass
    132 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 133     self.display_connect_table(input, **kwargs)
    134 elif isinstance(input, ConnectDataFrame):
    135     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/d

In [0]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType, LongType

# Define the main schema including the nested structure
Schema = StructType([StructField('id', IntegerType(), False),
                     StructField('country', StringType(), False),
                     StructField('description', StringType(), False),
                     StructField('input_timestamp', LongType(), False),
                     StructField('last_update_timestamp', LongType(), False),
                     StructField('source', StringType(), False),
                     StructField('user', StringType(), False)]
                     )

df_singList_recrd = spark.read \
    .format("json")\
    .schema(Schema)\
    .option('multiLine', True)\
    .option("mode", "PERMISSIVE")\
    .load("/Volumes/azureadb/pyspark/training/json/read_json/SingleLine_List/singlelineList_corrupt_record.json")

display(df_singList_recrd)
df_singList_recrd.printSchema()

id,country,description,input_timestamp,last_update_timestamp,source,user
1,IND,bravia,1124256609,1524256609,catalog,Hari
2,US,sony,1224256609,1424256609,SAP,Rajesh
3,CANADA,bse,1324256609,1524256609,ADLS,Lokesh
4,US,exchange,1424256609,1724256609,Blob,Sharath
5,SWEDEN,Stock,1524256609,1664256609,SQL,Sheetal
null,null,null,null,null,null,null


root
 |-- id: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- description: string (nullable = true)
 |-- input_timestamp: long (nullable = true)
 |-- last_update_timestamp: long (nullable = true)
 |-- source: string (nullable = true)
 |-- user: string (nullable = true)



In [0]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType, LongType

# Define the main schema including the nested structure
Schema = StructType([StructField('id', IntegerType(), False),
                     StructField('country', StringType(), False),
                     StructField('description', StringType(), False),
                     StructField('input_timestamp', LongType(), False),
                     StructField('last_update_timestamp', LongType(), False),
                     StructField('source', StringType(), False),
                     StructField('user', StringType(), False),
                     StructField("_corrupt_record", StringType(), True)]
                     )

df_singList_recrd = spark.read \
    .format("json")\
    .schema(Schema)\
    .option('multiLine', True)\
    .option("mode", "PERMISSIVE")\
    .load("/Volumes/azureadb/pyspark/training/json/read_json/SingleLine_List/singlelineList_corrupt_record.json")

display(df_singList_recrd)
df_singList_recrd.printSchema()

id,country,description,input_timestamp,last_update_timestamp,source,user,_corrupt_record
1,IND,bravia,1124256609,1524256609,catalog,Hari,null
2,US,sony,1224256609,1424256609,SAP,Rajesh,null
3,CANADA,bse,1324256609,1524256609,ADLS,Lokesh,null
4,US,exchange,1424256609,1724256609,Blob,Sharath,null
5,SWEDEN,Stock,1524256609,1664256609,SQL,Sheetal,null
null,null,null,null,null,null,null,"[ {""id"":1, ""source"":""catalog"", ""description"":""bravia"", ""input_timestamp"":1124256609, ""last_update_timestamp"":1524256609, ""country"":""IND"", ""user"":""Hari""}, {""id"":2, ""source"":""SAP"", ""description"":""sony"", ""input_timestamp"":1224256609, ""last_update_timestamp"":1424256609, ""country"":""US"", ""user"":""Rajesh""}, {""id"":3, ""source"":""ADLS"", ""description"":""bse"", ""input_timestamp"":1324256609, ""last_update_timestamp"":1524256609, ""country"":""CANADA"", ""user"":""Lokesh""}, {""id"":4, ""source"":""Blob"", ""description"":""exchange"", ""input_timestamp"":1424256609, ""last_update_timestamp"":1724256609, ""country"":""US"", ""user"":""Sharath""}, {""id"":5, ""source"":""SQL"", ""description"":""Stock"", ""input_timestamp"":1524256609, ""last_update_timestamp"":1664256609, ""country"":""SWEDEN"", ""user"":""Sheetal""}, {""id"":6, ""source"":""datawarehouse"", ""description"":""azure"", ""input_timestamp"":1624256609, ""last_update_timestamp"":1874256609, ""country"":""UK"", ""user"":""Raj"" {""id"":7, ""source"":""oracle"", ""description"":""ADF"", ""input_timestamp"":1779256609, ""last_update_timestamp"":188256609, ""country"":""Norway"", ""user"":""Synapse""}, {""id"":8, ""source"":""AZURE"", ""description"":""ETL"", ""input_timestamp"":1229256609, ""last_update_timestamp"":173256609, ""country"":""Norway"", ""user"":""Synapse""}, {""id"":9, ""source"":""AWS"", ""description"":""ELT"", ""input_timestamp"":1339256609, ""last_update_timestamp"":116256609, ""country"":""Norway"", ""user"":""Synapse""}, {""id"":10, ""source"":""GCC"", ""description"":""Git"", ""input_timestamp"":1569256609, ""last_update_timestamp"":129256609, ""country"":""Norway"", ""user"":""Synapse""} ]"


root
 |-- id: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- description: string (nullable = true)
 |-- input_timestamp: long (nullable = true)
 |-- last_update_timestamp: long (nullable = true)
 |-- source: string (nullable = true)
 |-- user: string (nullable = true)
 |-- _corrupt_record: string (nullable = true)



| JSON Type              | Correct Option              |
| ---------------------- | --------------------------- |
| One JSON per line      | default                     |
| Single-line JSON array (Single Line List) | `multiLine = true`          |
| Multi-line JSON array  | `multiLine = true`          |
| Nested JSON            | `multiLine = true` + schema |